# Time series performance benchmarks

The `titiler-cmr` API is deployed as a Lambda function in the SMCE VEDA AWS account. For small time series requests (<500 time points) you can expect a response from any of the endpoints within ~20 seconds. For larger time series requests, you run the risk of bumping into Lambda concurrency or timeout limits. This report shows some results from the `test_timeseries_benchmarks.py` script that sends many requests with varying time series lengths as well as several other parameters that affect runtime.

In [ ]:
import benchmark_analysis as ba

## xarray backend
The following tests use the following datasets to evaluate the limits of the `/timeseries` endpoints for the `xarray` backend 
- [GAMSSA 28km SST](https://podaac.jpl.nasa.gov/dataset/GAMSSA_28km-ABOM-L4-GLOB-v01): a daily 0.25 degree (~28 km) resolution dataset with sea surface temperature and sea ice fraction variables
- [MUR SST](https://cmr.earthdata.nasa.gov/search/concepts/C1996881146-POCLOUD.html): a daily 0.01 degree (~1km) resolution dataset with sea surface temperature variables

### statistics

Under the current deployment configuration `statistics` endpoint can process time series requests with up to ~1000 points. Requests that involve more than 1000 points are likely to fail.

In [ ]:
for dataset, df in ba.dfs["statistics"].items():
    fig = ba.plot_error_rate_heatmap(
        df=df,
        x="num_timepoints",
        y="bbox_dims",
        z="error_rate",
        labels={"x": "number of time points", "y": "bbox dimensions", "color": "error rate"},
        title=f"{dataset}: error rate by bbox size and number of time points",
    )
    fig.show()

In general, the size of the area you want to analyze will have minimal impact on the runtime! This is because `titiler.xarray` has to read the entire granule into memory before subsetting, so reducing the size of the AOI does not reduce the overall footprint of the computation.

In [ ]:
for dataset, df in ba.dfs["statistics"].items():
    ba.plot_line_with_error_bars(
        df=df.sort_values(["bbox_size", "num_timepoints"]),
        color="bbox_dims",
        title=f"{dataset}: statistics runtime",
    ).show()


### bbox (animations)

Under the current deployment configuration the `bbox` endpoint can reliably process time series requests with up to ~500 points. Requests that involve more than 500 points may fail if the area of interest is very large.

In [ ]:
for dataset, df in ba.dfs["bbox"].items():
    for img_size in sorted(df["img_size"].unique()):
        img_size_df = df[df["img_size"] == img_size]
        img_dims = img_size_df["img_dims"].unique()[0]
        ba.plot_error_rate_heatmap(
            df=img_size_df,
            x="num_timepoints",
            y="bbox_dims",
            z="error_rate",
            labels={"x": "number of time points", "y": "bbox dimensions", "color": "error rate"},
            title=f"{dataset}: image size {img_dims}",
        ).show()

The size of the area of interest increases the response time, especially for requests for higher resolution images.

In [ ]:
for dataset, df in ba.dfs["bbox"].items():
    ba.plot_line_with_error_bars(
        df=df.sort_values(["bbox_size", "num_timepoints"]),
        color="bbox_dims",
        facet_row="img_dims",
        title=f"{dataset}: runtime by bbox size and image dimensions"
    ).show()